In [ ]:
import cv2
import os
# get directory of the script

# continue going one folder up as long as folter name is not 'move_deve'
while os.path.basename(os.getcwd()) != 'photostim_deve':
    os.chdir('..')

move_deve_path = os.getcwd()

In [ ]:

experimenter = 'jm'
data_raw_dir = 'data_raw/'
data_vid_dir = 'data_vid/'

fps = 30  # Frames per second


In [ ]:
all_subject_dirs

In [ ]:
# get all 'input folders'

all_subject_dirs = sorted(os.listdir(data_raw_dir + experimenter))
# remove if not a directory
all_subject_dirs = [x for x in all_subject_dirs if os.path.isdir(data_raw_dir + experimenter + '/' + x)]

all_camera_dir = [] # this is where the single frame tiffs live
all_vid_path = [] # this is where we will save the videos

for subject_dir in all_subject_dirs:
    # get all subdirectories that end with _a
    all_session_dirs = sorted(os.listdir(data_raw_dir + experimenter + '/' + subject_dir))
    # now only take ones that end with '_a'
    all_session_dirs = [x for x in all_session_dirs if x[-2:] == '_s']

    # now if a session has a camera folder, add it to the list
    for session_dir in all_session_dirs:
        if os.path.isdir(data_raw_dir + experimenter + '/' + subject_dir + '/' + session_dir + '/camera'):
            all_camera_dir.append(data_raw_dir + experimenter + '/' + subject_dir + '/' + session_dir + '/camera')
            all_vid_path.append(data_vid_dir + experimenter + '/' + subject_dir + '/' + session_dir + '.avi')

for i in range(len(all_camera_dir)):
    print(all_camera_dir[i])
    print(all_vid_path[i])

In [ ]:
# now make a new folder structure in data_vid_dir
for i in range(len(all_vid_path)):
    # get the directory where the path is
    vid_dir = os.path.dirname(all_vid_path[i])
    os.makedirs(vid_dir, exist_ok=True)

In [ ]:
for (i, input_folder) in enumerate(all_camera_dir):

    if not os.path.exists(all_vid_path[i]):

        output_video = all_vid_path[i]

        output_size = None  # Set to (width, height) if resizing is needed
        # do avi for now
        fourcc = cv2.VideoWriter_fourcc(*'MJPG')  # Codec for AVI


        # Get list of images
        images = sorted([os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.tiff')])

        print('First 10 images:')
        for img in images[:10]:
            print(img)
        print('Last 10 images:')
        for img in images[-10:]:
            print(img)

        print('Number of images:', len(images))

        # Read the first image to get dimensions
        first_image = cv2.imread(images[0])
        height, width, _ = first_image.shape
        if output_size:
            width, height = output_size

        print("Video size: {}x{}".format(width, height))


        # Create VideoWriter
        video_writer = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

        # Write images to video
        for image_path in images:
            image = cv2.imread(image_path)
            if output_size:
                image = cv2.resize(image, output_size)
            video_writer.write(image)

            # print each 2000 frames
            if images.index(image_path) % 2000 == 0:
                print(f"Processed {images.index(image_path)} / {len(images)} frames")

        # Release VideoWriter
        video_writer.release()
        print(f"Video saved at {output_video}")
        
    else:
        print(f"Video {all_vid_path[i]} already exists, skipping...")